# 05 - Model Comparison and Conclusion
Compares the five tuned classifiers on the same untouched test set.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix

sns.set_style("whitegrid")
FIG_DIR = "../reports/figures"
PROC_DIR = "../data/processed"
os.makedirs(FIG_DIR, exist_ok=True)
def savefig(name):
    plt.savefig(f"{FIG_DIR}/{name}.png", dpi=150, bbox_inches="tight")

saved = joblib.load(f"{PROC_DIR}/results.pkl")
results, store, y_test = saved["results"], saved["store"], saved["y_test"]

## 1. Results table (all models)

In [ ]:
cols = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC", "TP", "FP", "FN", "TN"]
results[cols].round(4)

## 2. Tuned models only, ranked by PR-AUC

In [ ]:
tuned_only = results[results.index.str.contains("tuned")].sort_values("PR-AUC", ascending=False)
tuned_only[["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC", "FN", "FP"]].round(4)

In [ ]:
tuned_only.round(4).to_csv(f"{PROC_DIR}/final_results_table.csv")
metrics = ["Precision", "Recall", "F1", "PR-AUC"]
tuned_only[metrics].plot(kind="bar", figsize=(11, 5))
plt.xticks(rotation=25, ha="right"); plt.ylim(0, 1.05); plt.title("Tuned models: key metrics on the test set")
plt.legend(loc="lower right")
savefig("05_metric_bars"); plt.show()

## 3. ROC and Precision-Recall curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
for name, d in store.items():
    fpr, tpr, _ = roc_curve(y_test, d["score"])
    prec, rec, _ = precision_recall_curve(y_test, d["score"])
    auc_ = results.loc[f"{name} (tuned)", "ROC-AUC"]
    ap_ = results.loc[f"{name} (tuned)", "PR-AUC"]
    ax[0].plot(fpr, tpr, label=f"{name} (AUC={auc_:.3f})")
    ax[1].plot(rec, prec, label=f"{name} (AP={ap_:.3f})")
ax[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
ax[0].set_xlabel("False positive rate"); ax[0].set_ylabel("True positive rate"); ax[0].set_title("ROC curves"); ax[0].legend()
ax[1].axhline(y_test.mean(), color="k", ls="--", alpha=0.5, label="No-skill baseline")
ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].set_title("Precision-Recall curves"); ax[1].legend()
plt.tight_layout(); savefig("05_roc_pr_curves"); plt.show()

ROC curves look good for nearly every model because of the huge pool of true negatives. The Precision-Recall curves separate the models far better, which is why PR-AUC is the primary metric.

## 4. Best model

In [ ]:
best_name = tuned_only["PR-AUC"].idxmax()
b = tuned_only.loc[best_name]
print(f"Best by PR-AUC: {best_name}")
print(f"  Recall={b['Recall']:.3f}  Precision={b['Precision']:.3f}  F1={b['F1']:.3f}  PR-AUC={b['PR-AUC']:.3f}")
print(f"  Frauds caught: {int(b['TP'])} of {int(b['TP'] + b['FN'])} | False alarms: {int(b['FP'])}")

print("\nBest by Recall:", tuned_only["Recall"].idxmax())
print("Best by Precision:", tuned_only["Precision"].idxmax())
print("Best by F1:", tuned_only["F1"].idxmax())

In [ ]:
base = store[best_name.replace(" (tuned)", "")]
cm = confusion_matrix(y_test, base["pred"])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"])
plt.title(f"Confusion matrix: {best_name}"); plt.xlabel("Predicted"); plt.ylabel("Actual")
savefig("05_best_confusion"); plt.show()

## 5. Discussion (edit using your own numbers)
- **Accuracy is misleading:** the dummy model scores about 99.8% accuracy with zero recall.
- **Model choice:** the best model is selected by PR-AUC, then by recall at acceptable precision. State your winner and why, quoting the numbers above.
- **Trade-offs:**
  - Logistic Regression: fast and interpretable (odds ratios).
  - KNN: lazy learner, no training step, but slow prediction and sensitive to scaling.
  - Naive Bayes: very fast, assumes independent features (reasonable for PCA features), often higher recall but lower precision.
  - Decision Tree: readable rules, but can overfit without depth limits.
  - SVM: strong boundaries, but slow on large data (hence trained on the undersampled set).
- **Imbalance strategy:** undersampling raises recall but also false alarms; class weights keep more information.

## 6. Limitations
1. Only two days of data from one region (September 2013); fraud patterns change over time.
2. Features are PCA-anonymised, so business interpretation is limited.
3. Only about 470 fraud cases, so metrics have high variance; results depend on the split.
4. Undersampling discards most legitimate data.
5. Default 0.5 decision threshold used; in practice the threshold would be chosen by the cost of missed fraud vs false alarms.

## 7. Conclusion
Summarise in 3-4 lines: problem, best model with its recall, precision and PR-AUC, and what you would do next (more data, threshold tuning by cost, ensemble methods outside this syllabus).